# XAI-AMLBench: quick start

Generate a synthetic multi-hop money-laundering graph, train a GATv2 detector, compare it against baselines, and explain one flagged account with an audit-ready SAR narrative.

**Everything here is synthetic.** No real customer or transaction data is used or required.

Install (CPU only, no GPU needed):
```bash
pip install torch==2.4.1 --index-url https://download.pytorch.org/whl/cpu
pip install -e ".[gnn,xai]"
```

## 1. Generate a synthetic transaction graph

Five laundering typologies are embedded (smurfing, scatter-gather, cyclic loops, shell-company chains, cross-border velocity), alongside benign look-alike structures (large merchants, payroll, savings groups, escrow chains) so that the benchmark cannot be solved by amount or degree alone.

In [ ]:
from aml_synth import AMLGraphGenerator, GeneratorConfig

graph = AMLGraphGenerator(GeneratorConfig(n_accounts=5000, seed=1)).generate()
graph.summary()

## 2. Look at the data

`graph.accounts` and `graph.transactions` are plain pandas DataFrames.

In [ ]:
graph.accounts.head()

In [ ]:
graph.transactions.head()

In [ ]:
graph.transactions.typology.value_counts()

## 3. Build features and train a GATv2 detector

The graph is made bidirectional internally (every transaction also gets a reversed copy), so message passing reaches an account's counterparties in both directions -- otherwise a mule paying a central account could not be seen by that account at all. Training uses a stratified 60/20/20 split; the decision threshold is chosen on the validation split and reported on the held-out test split.

In [ ]:
from gnn_aml_core.train import arrays_from_frames, make_splits, fit_gnn

arrays = arrays_from_frames(graph.accounts, graph.transactions)
tr, va, te = make_splits(arrays["y"], seed=1)

result = fit_gnn(arrays, tr, va, te, model_name="gatv2", epochs=100, patience=6, seed=1)
result["report"]

## 4. Compare against the baselines

The project brief requires the GNN to beat Logistic Regression, XGBoost and Isolation Forest. We also compare against a strong graph-aware baseline (boosting on each account's own features plus its 1- and 2-hop neighbourhood averages) -- a GNN that cannot beat *that* has not shown that message passing helps, only that using the graph at all helps.

In [ ]:
from gnn_aml_core.baselines import evaluate_baselines

baselines = evaluate_baselines(arrays["x"], arrays["y"], tr, va, te,
                               edges=(arrays["src_tx"], arrays["dst_tx"]), seed=1)
import pandas as pd
pd.DataFrame({**{"GNN (GATv2)": result["report"]}, **baselines}).T[
    ["auc_roc", "pr_auc", "precision", "recall", "f1", "fpr"]]

For a paper-ready result, average this over several seeds:
```bash
python -m gnn_aml_core.benchmark --seeds 5
```
which writes `benchmarks/results.md` (a paste-ready table) and `benchmarks/results.json` (every number, every seed).

## 5. Explain one flagged account

Reduce the account's neighbourhood into plain-language facts, then generate a short, audit-ready narrative. `validate_narrative` guarantees every number, date and account id in the text is traceable back to the fact sheet -- the narrative can describe the evidence but can never invent evidence.

In [ ]:
import pandas as pd
from xai_explainer import build_facts, render_fact_sheet, template_narrative, validate_narrative

# In the running application this "explanation" dict comes from the GNNExplainer endpoint (/explain);
# here we build an equivalent one directly from one injected pattern, for a self-contained example.
pattern_id = graph.transactions.loc[graph.transactions.typology == "smurfing", "pattern_id"].iloc[0]
pattern = graph.transactions[graph.transactions.pattern_id == pattern_id]
accounts_by_id = graph.accounts.set_index("account_id")
involved = sorted(set(pattern.src) | set(pattern.dst))
# the subject is the hub of the pattern: the account with the most transactions in it
subject = pd.concat([pattern.src, pattern.dst]).value_counts().idxmax()

explanation = {
    "account_id": subject, "risk_score": 0.94, "reporting_threshold": 10_000,
    "nodes": [{"account_id": a, "account_type": accounts_by_id.loc[a, "account_type"],
               "country": accounts_by_id.loc[a, "country"]} for a in involved],
    "edges": [{"tx_id": r.tx_id, "src": r.src, "dst": r.dst, "amount": float(r.amount),
               "timestamp": int(r.timestamp), "cross_border": int(r.cross_border)}
              for r in pattern.itertuples()],
    "top_features": [{"feature": "near_thr_ratio", "weight": 0.4}, {"feature": "burst_6h", "weight": 0.2}],
}

facts = build_facts(explanation)
narrative = template_narrative(facts)
ok, problems = validate_narrative(narrative, render_fact_sheet(facts))
print(narrative)
print("\nvalidator:", "PASSED" if ok else problems)

## Next steps

- `aml_synth/README.md`, `gnn_aml_core/README.md`, `xai_explainer/README.md` -- module-level documentation.
- `python3 check_brief_v1.py` -- checks this repository against the project brief's acceptance criteria.
- `python3 check_stack_v1.py` -- checks that the full application (website, API, model, narrative service) is connected, if you have it running.
- The full application (a compliance-officer console) lives in `backend/`, `frontend/` and the Docker services described in the top-level `README.md`.